# Phase 3: Churn Prediction with XGBoost + SHAP

**Business question:** Which customers are about to be lost, and *why*?

**Two models, one pipeline** (`src/churn_model.py`):

| Model | Unit | Question | Why it matters |
|---|---|---|---|
| **Churn** | Customer | Will this customer buy again in the next 180 days? | Target retention spend at customers who can still be won |
| **Negative review** | Order | Will this delivered order get a 1–2 star review? | Lets support step in *before* a bad review and a lost customer |

**How it's kept honest**
- **No data leakage:** features use only information known at the snapshot date. A delivery or review only counts if it happened before the snapshot.
- **Out-of-time test:** the churn model trains on one snapshot and is tested on a later one; the review model trains on older orders and is tested on the most recent months.
- **Baseline comparison:** XGBoost is compared against logistic regression, so the extra complexity has to earn its place.

In [ ]:
import json
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))

import pandas as pd
from IPython.display import Image, display
from sqlalchemy import text

import churn_model as cm
from db import get_engine

engine = get_engine()
FIG = cm.FIG_DIR

## 1. Churn model
Because most Olist customers buy only once, the model is trained to find the rare
customers who **return**. Churn risk = 1 − P(return). ROC-AUC is identical either way.

Metrics to read:
- **ROC-AUC:** how well the model ranks returners above churners (0.5 = random).
- **PR-AUC:** precision/recall on the rare class; compare it with the positive rate, which is what random guessing would score.
- **Top-10% capture / lift:** if marketing contacts only the 10% highest-scored customers, what share of all returners do they reach?

In [ ]:
churn = cm.run("churn", horizon=180)
pd.DataFrame({k: churn[k] for k in ["xgboost", "logistic_regression"]})

In [ ]:
display(Image(FIG / "churn_roc.png", width=420))

### What drives it? (SHAP)
Each dot is a customer. Points to the right push the prediction toward *returning*;
colour shows whether the feature value was high (red) or low (blue).

In [ ]:
display(Image(FIG / "churn_shap_beeswarm.png", width=650))
display(Image(FIG / "churn_shap_groups.png", width=550))
pd.Series(churn["driver_group_share_pct"], name="% of explanation")

## 2. Negative-review model
Features describe the order at delivery time: how late it was, how long the carrier
took, price and freight, seller and customer location, category and payment method.

In [ ]:
review = cm.run("negative_review")
pd.DataFrame({k: review[k] for k in ["xgboost", "logistic_regression"]})

In [ ]:
display(Image(FIG / "negative_review_roc.png", width=420))
display(Image(FIG / "negative_review_shap_beeswarm.png", width=650))
display(Image(FIG / "negative_review_shap_groups.png", width=550))

## 3. Churn risk by customer segment
`churn_scores` holds a current risk score for every customer. Joining it with the
Phase 2 segments shows where retention budget should go.

In [ ]:
sql = '''
SELECT s.cluster_name,
       COUNT(*)                                   AS customers,
       ROUND(AVG(c.return_probability)::NUMERIC * 100, 2) AS avg_return_prob_pct,
       ROUND(100.0 * AVG((c.risk_tier = 'Likely to return')::INT), 1) AS likely_to_return_pct,
       ROUND(SUM(s.monetary)::NUMERIC, 0)         AS lifetime_revenue
FROM churn_scores c
JOIN customer_segments s USING (customer_unique_id)
GROUP BY s.cluster_name
ORDER BY lifetime_revenue DESC
'''
with engine.connect() as conn:
    by_segment = pd.read_sql(text(sql), conn)
by_segment

## 4. Results and recommendations
_Fill in once you run this on the full dataset._

**Churn model:** ROC-AUC __ on an out-of-time test (logistic regression: __). Contacting
the top 10% of scored customers reaches __% of all returners (__x lift over random).

**Negative-review model:** ROC-AUC __. Delivery-related features explain __% of the
model's predictions; orders delivered __+ days late are __x more likely to get a 1–2 star review.

**Recommendations**
1. Trigger a proactive apology or voucher when an order is predicted to arrive late.
2. Focus win-back spend on __ (the segment with the highest return probability × value).
3. Fix delivery performance in the worst states and categories first; it is the largest controllable driver.

**Interview notes:** why "return" is the positive class, why the test is out-of-time,
why there's no class re-weighting (it distorts probabilities), and what the SHAP
grouping does (share of mean |SHAP| by feature group).